In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import os
from torchvision.datasets import ImageFolder
from PIL import Image
from torch.utils.data import Dataset
import numpy as np
import matplotlib.pyplot as plt

In [19]:
# Enhanced Configuration
CONFIG = {
    "IMAGES_DIR_TRAINING": "./chest_xray/train",
    "IMAGES_DIR_VALIDATION": "./chest_xray/val",
    "CLASSES": ["NORMAL", "PNEUMONIA"],
    "BATCH_SIZE": 8,
    "IMAGE_SIZE": 256,  # Larger input size
    "N_EPOCHS": 30,
    "LEARNING_RATE": 1e-4,  # Reduced learning rate
    "WEIGHT_DECAY": 1e-4,
    "DROPOUT_RATE": 0.5,  # Added dropout
    "LABEL_SMOOTHING": 0.1,  # Prevent overconfidence
    "PATIENCE": 5 ,
    "NUM_CLASSES": 2,
}


In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Num GPUs Available:", torch.cuda.device_count())

Num GPUs Available: 1


In [21]:
from PIL import Image

class PneumoniaDataset(Dataset):

    def __init__(self, image_dir, transform=None):
        self.image_paths = []
        self.transform = transform

        # Traverse directories and collect all image file paths
        for subdir, _, files in os.walk(image_dir):
            for file in files:
                # Build full file path
                self.image_paths.append(os.path.join(subdir, file))

    def __getitem__(self, index):
        # allow indexing
        img_path = self.image_paths[index]
        # img_path = os.path.join(self.image_dir, self.image_files[index])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        # Extract class from folder name
        label_name = os.path.basename(os.path.dirname(img_path))
        label = CONFIG["CLASSES"].index(label_name)

        return image, label
    
    def __len__(self):
        # len (dataset)
        return len(self.image_paths)


In [22]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((CONFIG["IMAGE_SIZE"], CONFIG["IMAGE_SIZE"])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((CONFIG["IMAGE_SIZE"], CONFIG["IMAGE_SIZE"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [23]:
from torch.utils.data import Dataset,DataLoader

# Load datasets
train_dataset = PneumoniaDataset(CONFIG["IMAGES_DIR_TRAINING"], train_transform)
val_dataset = PneumoniaDataset(CONFIG["IMAGES_DIR_VALIDATION"], val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["BATCH_SIZE"],
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["BATCH_SIZE"],
    shuffle=False,  # Important: no shuffle for validation
)

In [25]:
import gc

# clear up the leftover memory
torch.cuda.empty_cache()
gc.collect()

5468

In [26]:
# Enhanced Data Augmentation
train_transform = transforms.Compose([
    transforms.Resize((CONFIG["IMAGE_SIZE"], CONFIG["IMAGE_SIZE"])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),  # Added
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # Stronger
    transforms.RandomAdjustSharpness(sharpness_factor=2),  # Added
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Enhanced Model Architecture
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(CONFIG["DROPOUT_RATE"]),  # Added dropout
    nn.Linear(num_ftrs, 512),  # Added hidden layer
    nn.ReLU(),
    nn.BatchNorm1d(512),  # Added batchnorm
    nn.Linear(512, CONFIG["NUM_CLASSES"])
)
model = model.to(device)

# Enhanced Loss Function
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["LABEL_SMOOTHING"])

# Enhanced Optimizer
optimizer = optim.AdamW(model.parameters(), 
                       lr=CONFIG["LEARNING_RATE"], 
                       weight_decay=CONFIG["WEIGHT_DECAY"])

# Enhanced Learning Rate Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=2, factor=0.5)

# Early Stopping
best_val_f1 = 0
patience_counter = 0

for epoch in range(CONFIG["N_EPOCHS"]):
    model.train()
    train_loss = 0.0
    train_preds, train_labels = [], []
    
    for images, labels in tqdm(train_loader, desc=f'Train Epoch {epoch+1}'):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        train_preds.extend(preds.detach().cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

    # Validation
    model.eval()
    val_loss = 0.0
    val_preds, val_labels = [], []
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Val Epoch {epoch+1}'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    train_loss /= len(train_loader)
    train_acc = accuracy_score(train_labels, train_preds)
    train_f1 = f1_score(train_labels, train_preds, average='binary')
    
    val_loss /= len(val_loader)
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds, average='binary')
    
    # Update scheduler
    scheduler.step(val_f1)  # Monitor F1 instead of loss
    
    print(f"\nEpoch {epoch+1}/{CONFIG['N_EPOCHS']}")
    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")

# Load best model
model.load_state_dict(torch.load("best_model.pth"))
torch.save(model, "pneumonia_resnet50_final.pth")

Train Epoch 1:   0%|          | 0/652 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 398.30 MiB is allocated by PyTorch, and 39.70 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)